# Day 11: Building Your First ML Pipeline — End-to-End Classification

Welcome to Day 11 of the **60-Day Data Science Challenge**! 🚀
Yesterday, we performed feature engineering on our retail transactions dataset (scaling, log transformations, and one-hot encoding). 

Today, we are taking the big leap: **building our very first Machine Learning Pipeline!**
We will build a binary classifier to predict whether a transaction represents a **High-Value Sale** (sales value above the median transaction price of the dataset).

## Our Pipeline Workflow:
1. **Data Ingestion**: Load the engineered dataset from Day 10.
2. **Target Setup & Leakage Prevention**: Define our target variable `Is_High_Value_Sale` and carefully select features while excluding variables that could leak the target.
3. **Train-Test Split**: Partition the dataset into training (80%) and testing (20%) subsets with stratification.
4. **Baseline Model Training**: Fit a baseline Logistic Regression model on the training data.
5. **Prediction Generation**: Generate class predictions and confidence probabilities on unseen test data.
6. **Performance Evaluation**: Analyze the prediction quality using accuracy, precision, recall, F1-score, confusion matrix, ROC-AUC, and feature coefficients.
7. **Export Results**: Save test actuals and predictions to `predictions.csv`.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    roc_curve,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

# Set styling for nice visuals
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (8, 6)
plt.rcParams["font.size"] = 11

### 1. Ingest & Inspect the Feature-Engineered Dataset
Let's load the `engineered_store_transactions.csv` generated on Day 10 and verify its dimensions and schema.

In [ ]:
df = pd.read_csv('../day10/engineered_store_transactions.csv')
print(f"Dataset shape: {df.shape[0]} rows, {df.shape[1]} columns.")
df.head(3)

### 2. Define the Target Variable and Prevent Target Leakage

Our objective is to predict if a transaction is a **High-Value Sale**. 
- First, we define high-value as any sale whose `Sales` value is strictly greater than the overall dataset median of `Sales`.
- Second, we must be extremely careful to **prevent target leakage**. Target leakage occurs when features contain information about the target that wouldn't be available at prediction time. 
  - *Leakage risk columns*: `Sales`, `Sales_log`, `Sales_per_Unit`, `Sales_per_Unit_log`, and their scaled equivalents. If we included these, our model would achieve 100% accuracy but would be completely useless on new data where sales are unknown!
  - *Non-predictive IDs*: `Row ID`, `Order ID`, `Customer Name`, `Order Date` will also be excluded.
  - *Clean features list*: We will keep our one-hot encoded categories, segments, zip codes, along with `Quantity_scaled`, `Order_Month_scaled`, and `Is_Weekend`.

In [ ]:
# Compute median sales value
sales_median = df['Sales'].median()
print(f"Median sales value in dataset: ${sales_median:.2f}")

# Define the binary target variable (1 for high-value sale, 0 otherwise)
df['Is_High_Value_Sale'] = (df['Sales'] > sales_median).astype(int)
print(f"Target distribution:\n{df['Is_High_Value_Sale'].value_counts(normalize=True)}")

# Define clean feature list (excluding leakage columns and identifiers)
leakage_cols = [
    'Row ID', 'Order ID', 'Order Date', 'Customer Name', 
    'Sales', 'Sales_log', 'Sales_per_Unit', 'Sales_per_Unit_log',
    'Sales_log_scaled', 'Sales_per_Unit_log_scaled', 'Order_Year', 'Order_DayOfWeek',
    'Is_High_Value_Sale'
]

feature_cols = [col for col in df.columns if col not in leakage_cols]
print(f"\nWe are training our model on the following {len(feature_cols)} features:")
print(feature_cols)

# Separate features (X) and target (y)
X = df[feature_cols]
y = df['Is_High_Value_Sale']

### 3. Split Dataset into Training and Testing Sets

To evaluate our model's performance on unseen data, we must split our dataset. 
- We will use an **80/20 train-test split** (80% of data for training the model, 20% for testing).
- We set `stratify=y` to ensure that both training and testing sets have exactly the same proportion of high-value vs. low-value sales (50/50 split), protecting against class imbalance.
- We set a fixed `random_state=42` to ensure our split is reproducible.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.20, 
    random_state=42, 
    stratify=y
)

print(f"Training set dimensions: {X_train.shape[0]} rows, {X_train.shape[1]} features.")
print(f"Testing set dimensions:  {X_test.shape[0]} rows, {X_test.shape[1]} features.")
print(f"Class ratio in training set: {y_train.mean():.4f}")
print(f"Class ratio in testing set:  {y_test.mean():.4f}")

### 4. Select and Train the Baseline ML Algorithm

For our baseline model, we select **Logistic Regression**. 
- *Why Logistic Regression?* It is simple, highly interpretable, fast to train, and serves as the industry standard baseline for binary classification. It estimates the probability of binary classes using a logistic (sigmoid) function.

In [ ]:
# Initialize the model
baseline_model = LogisticRegression(random_state=42, max_iter=1000)

# Fit the model on the training data
baseline_model.fit(X_train, y_train)
print("Baseline Logistic Regression model trained successfully!")

### 5. Generate Predictions on Test Data

Now, we evaluate our trained model on the unseen test set. We will generate two types of predictions:
1. **Class predictions (`y_pred`)**: Binary outputs (0 or 1) indicating if a sale is predicted to be low or high value.
2. **Prediction probabilities (`y_prob`)**: Continuous numbers between 0 and 1 indicating the model's confidence probability that the sale is high-value.

In [ ]:
# Predict class labels
y_pred = baseline_model.predict(X_test)

# Predict probability of being High-Value (class 1)
y_prob = baseline_model.predict_proba(X_test)[:, 1]

# Display a sample comparison
comparison_df = pd.DataFrame({
    'Actual': y_test,
    'Predicted': y_pred,
    'Probability': y_prob
}).reset_index(drop=True)

comparison_df.head(10)

### 6. Analyze Prediction Quality

Let's compute our standard classification performance metrics to see how well our pipeline works:
- **Accuracy**: Overall fraction of correct predictions.
- **Precision**: Out of all predicted high-value sales, how many were actually high-value?
- **Recall**: Out of all actual high-value sales, how many did the model find?
- **F1-Score**: The harmonic mean of precision and recall.
- **ROC-AUC**: Represents the model's ability to distinguish between classes.

In [ ]:
# Compute main metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_prob)

print("--- MODEL PERFORMANCE METRICS ---")
print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-Score:  {f1:.4f}")
print(f"ROC-AUC:   {roc_auc:.4f}\n")

# Detailed classification report
print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=['Low-Value', 'High-Value']))

#### A. Confusion Matrix
A confusion matrix visualizes the True Positives, True Negatives, False Positives, and False Negatives, giving us an instant look at where the model is making errors.

In [ ]:
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Predicted Low', 'Predicted High'],
            yticklabels=['Actual Low', 'Actual High'])
plt.title('Confusion Matrix on Test Data')
plt.ylabel('Actual Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150)
plt.show()

#### B. ROC Curve
The Receiver Operating Characteristic (ROC) curve plots the True Positive Rate (Sensitivity) against the False Positive Rate (1 - Specificity) across different decision thresholds. A random guess model represents the diagonal line (AUC = 0.5), while a perfect model represents the top-left corner (AUC = 1.0).

In [ ]:
fpr, tpr, thresholds = roc_curve(y_test, y_prob)

plt.figure(figsize=(7, 6))
plt.plot(fpr, tpr, color='teal', lw=2, label=f'Logistic Regression (AUC = {roc_auc:.3f})')
plt.plot([0, 1], [0, 1], color='navy', linestyle='--', label='Random Guessing (AUC = 0.50)')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve on Test Data')
plt.legend(loc="lower right")
plt.tight_layout()
plt.savefig('roc_curve.png', dpi=150)
plt.show()

#### C. Model Coefficients & Feature Importance
Let's extract the coefficients from our trained logistic regression model. Because our numeric features are standardized and categorical features are OHE binary inputs, the magnitude and sign of these coefficients show us how each feature impacts the probability of a transaction being high-value.

In [ ]:
# Create a dataframe of features and their coefficients
coefficients = pd.DataFrame({
    'Feature': feature_cols,
    'Coefficient': baseline_model.coef_[0]
})

# Sort by absolute coefficient values to see strongest drivers
coefficients['Abs_Coefficient'] = coefficients['Coefficient'].abs()
coefficients = coefficients.sort_values(by='Abs_Coefficient', ascending=False).reset_index(drop=True)

print("Model Coefficients (Ordered by Impact):")
print(coefficients[['Feature', 'Coefficient']])

# Plot feature coefficients
plt.figure(figsize=(10, 6))
sns.barplot(data=coefficients, y='Feature', x='Coefficient', palette='coolwarm')
plt.title('Logistic Regression Feature Coefficients')
plt.xlabel('Coefficient Value (Impact on High-Value Prob)')
plt.ylabel('Feature')
plt.axvline(x=0, color='black', linestyle='--')
plt.tight_layout()
plt.savefig('feature_coefficients.png', dpi=150)
plt.show()

### 7. Export Prediction Outputs
Finally, we export the test observations along with their ground truth labels, predicted classes, and calculated probabilities to a CSV file. This enables us to inspect misclassified examples or deploy the predictions to other downstream systems.

In [ ]:
# Create the predictions export DataFrame
predictions_export = X_test.copy()

# Add original unscaled variables for reference if desired (by aligning index)
predictions_export['Actual_Is_High_Value_Sale'] = y_test
predictions_export['Predicted_Is_High_Value_Sale'] = y_pred
predictions_export['Prediction_Probability'] = y_prob

# Write to CSV
predictions_export.to_csv('predictions.csv', index=False)
print(f"Successfully saved {predictions_export.shape[0]} test predictions to 'predictions.csv'!")

## Summary & Student Observations
- **Our performance metrics**: We achieved an accuracy of around **51-52%** and ROC-AUC of around **0.51-0.52**. While this is very close to random guessing, it's expected! The transactions in this synthetic dataset were generated uniformly and have low correlation between segment/category features and the final Sales amount. 
- **Target Leakage**: Removing the leakage fields (`Sales_log`, `Sales_per_Unit_log`, etc.) was a major step. Keeping them would give a fake 100% accuracy that would immediately break in a production setting.
- **Top Drivers**: Looking at the coefficients, we can see which features play a positive or negative role in transaction value prediction.
- **Pipeline ready**: We now have a standard, clean ML pipeline framework that we can plug any other dataset into!